# DBSCAN

*Density-Based Spatial Clustering of Applications with Noise*. Algoritmo di clustering basato su densità.

M.Ester, H.Kriegel, J.Sander, X.Xu. *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise*. 1996.

Parametri:

- `min_pts`: numero minimo di punti per formare un cluster
- `eps`: distanza massima affinché due punti siano considerati vicini.

Numero di cluster non fissato a priori, viene deciso dall'algoritmo.

## Definizioni

- *Core point*: un punto che ha almeno `min_pts` in un suo intorno `eps`.
- *Border point*: un punto non core ma che è vicino ad un core
- *Noise point*: nè core, nè border

![image-2.png](materiale_2026/08-01-dbscan_connessioni.png)


### Raggiungibilià

- Un punto `p` è *raggiungibile direttamente per densità* da `q` se `q` è un core e `p` è nel suo vicinato (fugura a sinistra). *In questo caso `q` appartiene ad un cluster*.
- Un punto `p` è *raggiungibile per densità* da `q` se esiste una catena di punti core a partire da `q` ciascuno raggiungibile direttamente per densità dal precedente (nella figura al centro, la catena contiene solo due punti core). *In questo caso tutti i core della catena sono punti di un unico cluster.*
- I punti `p` e `q` sono *connessi per densità*  se esiste un punto `u` da cu `p` e `q` sono raggiungibili per densità (figura a destra).

## Cluster

`C` è un cluster, rispetto `min_pts` e `eps` se è non vuoto e soddisfa le seguenti proprietà:
- Per ogni coppia `p` e `q`, se `q` è in `C` e `p` è raggiungibile per densità da `q`, anche `p` appartiene a `C`.
- Ogni coppia `p` e `q` in `C` deve essere connessa per densità. (Ovvero solo i punti raggiungibili per densità da un core in `C` sono in `C`)

Dalla definizione, i border possono appartenere a più cluster perché raggiungibili per densità da più core non vicini. L'algoritmo ne sceglierà uno e sarà il primo da cui viene raggiunto.

## L'Algoritmo

- INPUT: un insieme `D` di `n` punti, `eps`, `min_pts`
- OUTPUT: per ogni punto `p` in `D`, un valore `label[p]` in `{Noise, 0, 1, ...}`

* Inizializzazione: `label[p] ← non_visitato` per tutti i punti `p`
* Per ogni punto `p` in `D` etichettato `non_visitato`:
    1. Sia `N` l'insieme dei punti a distanza `≤ eps` da `p`
    2. Se `|N| < min_pts`, non è un core point, ma potrebbe diventare border. Per il momento
        - `label[p] ← Noise`
    3. Altrimenti `p` è un core di un nuovo cluster `c`, quindi
        * `label[p] ← c`
        * `S ← N - {p}`
        * Prosegui prelevando punti `q` da `S` fino ad esaurimento di quest'ultimo
            - Se `q` è già in un cluster, scartalo (se `label[q]` è un numero)
            - `label[q] ← c` anche se valeva `Noise`
            - Se `q` è un core point
                - Definisci `N` come l'insieme dei vicini di `q`
                - `S ← S ∪ (N - {q})`

## Significato dei parametri

DBSCAN dipende principalmente da due parametri:

```text
eps:
raggio del vicinato

min_pts:
numero minimo di punti richiesti nel vicinato
```

Dato un punto `p`, il suo vicinato è l'insieme dei punti che distano al massimo `eps` da `p`.

Se nel vicinato di `p` ci sono almeno `min_pts` punti, allora `p` è un core point.

Quindi:

```text
eps grande:
più punti risultano vicini
cluster più grandi
meno punti noise

eps piccolo:
meno punti risultano vicini
cluster più frammentati
più punti noise

min_pts grande:
serve maggiore densità per formare un cluster

min_pts piccolo:
è più facile formare cluster
```

## Nota su `min_pts`

A seconda della convenzione, `min_pts` può essere interpretato in due modi:

```text
1. numero minimo di punti vicini escluso il punto stesso
2. numero minimo di punti nel vicinato incluso il punto stesso
```

Nelle implementazioni più comuni, come `sklearn`, il parametro corrispondente si chiama `min_samples` e include anche il punto stesso.

Quindi, se:

```python
min_samples = 5
```

un punto è core se nel suo vicinato `eps` ci sono almeno 5 punti contando anche lui.

Questo dettaglio è importante quando si confronta la definizione teorica con il codice.

## Interpretazione operativa dei punti

DBSCAN assegna a ogni punto uno dei tre ruoli:

```text
core:
ha abbastanza vicini nel raggio eps.
Può far partire o espandere un cluster.

border:
non ha abbastanza vicini per essere core,
ma si trova nel vicinato di un core.
Appartiene a un cluster, ma non lo espande.

noise:
non è core e non è nel vicinato di nessun core.
Viene considerato rumore o outlier.
```

La differenza importante è:

```text
i core espandono i cluster
i border vengono assegnati a un cluster ma non lo espandono
i noise restano fuori dai cluster
```

## Intuizione sulla raggiungibilità

La raggiungibilità serve a capire come DBSCAN espande un cluster.

Un punto `p` è direttamente raggiungibile da `q` se:

```text
q è core
p è nel vicinato eps di q
```

Quindi un core point può "agganciare" altri punti al cluster.

La raggiungibilità per densità estende questa idea a catena:

```text
q core raggiunge p1
p1 core raggiunge p2
p2 core raggiunge p3
...
```

Tutti i core collegati da questa catena appartengono allo stesso cluster.

I border possono essere inclusi nel cluster, ma non possono continuare la catena perché non sono core.

## Border point tra due cluster

Un border point può trovarsi nel vicinato di core point appartenenti a cluster diversi.

In teoria potrebbe essere compatibile con più cluster.

Nell'algoritmo DBSCAN, però, viene assegnato al primo cluster che lo raggiunge durante la scansione dei punti.

Quindi l'assegnazione di alcuni border point può dipendere dall'ordine con cui l'algoritmo visita i punti.

I core point, invece, determinano in modo più stabile la struttura principale dei cluster.

## Perché un punto marcato `Noise` può poi entrare in un cluster

Quando DBSCAN visita un punto `p` e trova che non è core, lo marca temporaneamente come `Noise`.

Questo non significa necessariamente che resterà noise fino alla fine.

Più avanti, un altro punto core potrebbe avere `p` nel proprio vicinato.

In quel caso `p` diventa un border point e viene assegnato al cluster del core.

Quindi l'etichetta `Noise` può essere provvisoria.

Alla fine dell'algoritmo restano noise solo i punti che non sono né core né vicini ad alcun core.

## Complessità computazionale

Il costo di DBSCAN dipende da come vengono cercati i vicini entro distanza `eps`.

Nel caso più semplice, per ogni punto si confronta la distanza con tutti gli altri punti.

Con `n` punti, questo porta a un costo circa:

$$
O(n^2)
$$

Usando strutture dati per nearest neighbors, come KDTree o BallTree, la ricerca dei vicini può essere accelerata.

In quel caso il costo può migliorare, soprattutto in dimensioni basse o moderate.

Se però il numero di feature è molto alto, queste strutture diventano meno efficaci e il costo può avvicinarsi nuovamente al caso quadratico.

## Vantaggi e limiti di DBSCAN

DBSCAN ha alcuni vantaggi importanti:

```text
non richiede di scegliere a priori il numero di cluster
riconosce cluster di forma non sferica
identifica esplicitamente punti noise/outlier
```

Funziona bene quando i cluster sono separati da regioni a bassa densità.

Ha però anche alcuni limiti:

```text
è sensibile alla scelta di eps e min_pts
fatica con cluster aventi densità molto diverse
può funzionare peggio in alta dimensionalità
```

Se due cluster hanno densità molto diversa, un singolo valore di `eps` può essere troppo grande per il cluster denso e troppo piccolo per quello sparso.

## Confronto con k-means

DBSCAN e k-means risolvono entrambi problemi di clustering, ma hanno assunzioni diverse.

```text
k-means:
richiede di fissare k
tende a trovare cluster compatti e circa sferici
assegna ogni punto a un cluster
è sensibile agli outlier

DBSCAN:
non richiede di fissare il numero di cluster
trova cluster basati sulla densità
può trovare cluster di forma arbitraria
può lasciare alcuni punti come noise
```

Quindi DBSCAN è più adatto quando i cluster hanno forme irregolari e quando vogliamo individuare outlier.

k-means è più adatto quando ci aspettiamo cluster compatti e ben separati.